# Improved Music Genre Classification
This notebook implements an Artificial Neural Network (ANN) with Learning Rate Scheduling and Batch Normalization.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import ReduceLROnPlateau
import warnings
warnings.filterwarnings('ignore')

In [ ]:
data = pd.read_csv('features_3_sec.csv')
data = data.drop(['filename', 'length'], axis=1)
data.head()

In [ ]:
X = data.drop(['label'], axis=1)
y = data['label']

le = LabelEncoder()
y_encoded = le.fit_transform(y)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42
)
print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")

In [ ]:
y_train_cat = to_categorical(y_train)
y_test_cat = to_categorical(y_test)
num_classes = y_train_cat.shape[1]

model = Sequential([
    Dense(256, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(128, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),

    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),

    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

history = model.fit(
    X_train, y_train_cat,
    epochs=80,
    batch_size=32,
    validation_data=(X_test, y_test_cat),
    callbacks=[lr_scheduler],
    verbose=1
)

In [ ]:
ann_loss, ann_acc = model.evaluate(X_test, y_test_cat, verbose=0)
y_pred_ann = np.argmax(model.predict(X_test, verbose=0), axis=1)

print(f"\nANN Test Accuracy: {ann_acc:.4f} ({ann_acc*100:.2f}%)")
print(f"Best val_loss:    {min(history.history['val_loss']):.4f}")
print(f"\nClassification Report:\n")
print(classification_report(y_test, y_pred_ann, target_names=le.classes_))